# INFOSYS 722 Iteration 4 – BDAS
Diabetes prediction on **AWS EC2 + Apache Spark + PySpark + Spark MLlib**.
Pipeline mirrors `src/`: data → preparation → transformation → modelling → evaluation.
Run from the project root (`INFOSYS722-BDAS-Diabetes/`).


In [1]:
import sys
sys.path.insert(0, "src")
from spark_session import get_spark
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
spark = get_spark()
print("Spark Version:", spark.version)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/ubuntu/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


26/09/23 10:54:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark Version: 4.2.0


## Step 2 – Data Understanding
Load D1a as a distributed Spark DataFrame and verify records, schema and statistics.


In [2]:
df = (spark.read.option("header", True).option("inferSchema", True)
      .csv("data/brfss2023_diabetes_analysis.csv"))
print("Records:", df.count())
df.printSchema()
df.describe().show()


Records: 433323
root
 |-- ID: integer (nullable = true)
 |-- Diabetes_binary: integer (nullable = true)
 |-- HighBP: integer (nullable = true)
 |-- HighChol: integer (nullable = true)
 |-- CholCheck: integer (nullable = true)
 |-- BMI: double (nullable = true)
 |-- Smoker: integer (nullable = true)
 |-- Stroke: integer (nullable = true)
 |-- HeartDiseaseorAttack: integer (nullable = true)
 |-- PhysActivity: integer (nullable = true)
 |-- HvyAlcoholConsump: integer (nullable = true)
 |-- AnyHealthcare: integer (nullable = true)
 |-- NoDocbcCost: integer (nullable = true)
 |-- GenHlth: integer (nullable = true)
 |-- MentHlth: integer (nullable = true)
 |-- PhysHlth: integer (nullable = true)
 |-- DiffWalk: integer (nullable = true)
 |-- Sex: integer (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Education: integer (nullable = true)
 |-- Income: integer (nullable = true)



26/09/23 10:55:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------------+-----------------+-----------------+-------------------+-------------------+------------------+------------------+------------------+
|summary|                ID|    Diabetes_binary|            HighBP|           HighChol|         CholCheck|               BMI|            Smoker|             Stroke|HeartDiseaseorAttack|       PhysActivity|  HvyAlcoholConsump|      AnyHealthcare|        NoDocbcCost|           GenHlth|         MentHlth|         PhysHlth|           DiffWalk|                Sex|               Age|         Education|            Income|
+-------+------------------+-------------------+------------------+-------------------+------------------+------------------+------------------+-------------------+

## Step 3 – Data Preparation
Drop target-missing records, resolve remaining blanks, persist cleaned Parquet.


In [3]:
target = "Diabetes_binary"
print("Before cleaning:", df.count())
df_clean = df.dropna(subset=[target])
df_clean = df_clean.fillna(0)
print("After cleaning:", df_clean.count())
df_clean.write.mode("overwrite").parquet("data/clean_diabetes.parquet")
print("Saved: clean_diabetes.parquet")


Before cleaning: 433323


After cleaning: 432339


Saved: clean_diabetes.parquet


## Step 4 – Data Transformation
Assemble the 19 cleaned predictors into a Spark MLlib `features` vector.


In [4]:
df = spark.read.parquet("data/clean_diabetes.parquet")
features = [c for c in df.columns if c != target and c != "ID"]
print("Number of features:", len(features))
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_model = assembler.transform(df).select("features", target)
df_model.show(5)
df_model.write.mode("overwrite").parquet("data/model_ready.parquet")
print("Saved: model_ready.parquet")


Number of features: 19


+--------------------+---------------+
|            features|Diabetes_binary|
+--------------------+---------------+
|(19,[0,2,3,9,11,1...|              1|
|(19,[0,1,2,3,7,9,...|              0|
|[1.0,1.0,1.0,22.3...|              0|
|(19,[2,3,7,9,11,1...|              0|
|(19,[0,2,3,7,9,11...|              1|
+--------------------+---------------+
only showing top 5 rows


Saved: model_ready.parquet


## Steps 5–6 – Method & Algorithm Selection
**Binary classification** with Spark MLlib: Logistic Regression (linear scorer) and Random Forest, 100 trees (distributed benchmark).


In [5]:
df_model = spark.read.parquet("data/model_ready.parquet")
df_model = df_model.withColumnRenamed(target, "label")
train, test = df_model.randomSplit([0.8, 0.2], seed=42)
print("Training:", train.count())
print("Testing:", test.count())


Training: 345775


Testing: 86564


In [6]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train)
lr_prediction = lr_model.transform(test)
lr_prediction.select("label", "prediction", "probability").show(10)
lr_prediction.write.mode("overwrite").parquet("output/lr_prediction")


+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    1|       1.0|[0.32525023929747...|
|    0|       0.0|[0.84618445840142...|
|    0|       0.0|[0.82040303123798...|
|    0|       0.0|[0.66770815501883...|
|    1|       1.0|[0.44474505600219...|
|    1|       0.0|[0.63836806465413...|
|    1|       0.0|[0.81113963563443...|
|    0|       0.0|[0.52836861316697...|
|    0|       0.0|[0.52873134769057...|
|    1|       1.0|[0.48369927764412...|
+-----+----------+--------------------+
only showing top 10 rows


In [7]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, seed=42)
rf_model = rf.fit(train)
rf_prediction = rf_model.transform(test)
rf_prediction.select("label", "prediction", "probability").show(10)
rf_prediction.write.mode("overwrite").parquet("output/rf_prediction")
print("Model output saved")


+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    1|       0.0|[0.65539821419851...|
|    0|       0.0|[0.82513847803964...|
|    0|       0.0|[0.82513847803964...|
|    0|       0.0|[0.80755216973690...|
|    1|       0.0|[0.69165696169914...|
|    1|       0.0|[0.74772911887271...|
|    1|       0.0|[0.82240212783453...|
|    0|       0.0|[0.80339707894357...|
|    0|       0.0|[0.79923431882478...|
|    1|       0.0|[0.78263882628255...|
+-----+----------+--------------------+
only showing top 10 rows


Model output saved


## Step 8 – Interpretation & Evaluation
Confusion matrix, AUC and accuracy for the Random Forest on the held-out partition.


In [8]:
prediction = spark.read.parquet("output/rf_prediction")
prediction.groupBy("label", "prediction").count().show()
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(prediction)
accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(prediction)
print("AUC:", auc)
print("Accuracy:", accuracy)
import pandas as pd
pd.DataFrame({"model": ["Random Forest"], "AUC": [auc], "Accuracy": [accuracy]}).to_csv("output/bdas_metrics.csv", index=False)
print("Saved: bdas_metrics.csv")


+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    1|       0.0|13920|
|    0|       0.0|72341|
|    1|       1.0|  202|
|    0|       1.0|  101|
+-----+----------+-----+



AUC: 0.789824619830455
Accuracy: 0.8380273554826487
Saved: bdas_metrics.csv


In [9]:
spark.stop()
